In [2]:
!pip uninstall -y llama-cpp-python
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu122
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 GB 794.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.6 MB/s eta 0:00:00


In [28]:
from llama_cpp import Llama

llm = Llama.from_pretrained(
    repo_id="mradermacher/ChemLLM-20B-Chat-SFT-i1-GGUF",
    filename="ChemLLM-20B-Chat-SFT.i1-Q4_K_M.gguf",
    n_ctx=2048,
    n_gpu_layers=25
)

./ChemLLM-20B-Chat-SFT.i1-Q4_K_M.gguf:   0%|          | 0.00/12.0G [00:00<?, ?B/s]

llama_model_loader: loaded meta data with 41 key-value pairs and 435 tensors from /root/.cache/huggingface/hub/models--mradermacher--ChemLLM-20B-Chat-SFT-i1-GGUF/snapshots/1624398a3921d8ec22c7b4513d0b346a1c40b78b/./ChemLLM-20B-Chat-SFT.i1-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = internlm2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Internlm2 Chat 20b
llama_model_loader: - kv   3:                       general.organization str              = Internlm
llama_model_loader: - kv   4:                           general.finetune str              = chat
llama_model_loader: - kv   5:                           general.basename str              = internlm2
llama_model_l

In [30]:
import json

TARGET_SMILES = "O=[N+]([O-])c1ccccc1"

SYSTEM_PROMPT = """
You are an expert Organic Chemist specializing in Retrosynthesis.
Your task is to propose practical alternative forward synthesis routes that make the given Target Molecule.

CRITICAL RULES:
1. Output MUST be valid JSON only.
2. Use standard SMILES strings for all molecules.
3. Verify that each route can plausibly form the stated final product.
4. Do not output Markdown formatting like ```json.
5. Use literature-like reaction conditions. If exact values are uncertain, provide realistic approximate ranges.
6. Generate exactly 1 distinct route option.
7. Each route must contain exactly one reaction step in its steps list. Do not propose intermediates, precursor syntheses, or multi-step sequences.
8. The only step product_smiles in every route MUST be the target molecule SMILES after canonicalization.
9. Each route must be genuinely distinct: different disconnection, starting material class, or key reaction type.
10. Avoid ambiguous reagent descriptions: if SMILES CO is methanol, call it methanol or MeOH in conditions; only use carbon monoxide when the intended reagent is truly carbon monoxide.
11. The named or descriptive reaction must include the catalysts, activators, bases, acids, or other reagents that make the stated chemistry plausible.
12. Temperature values must include units or clear descriptors such as room temperature, reflux, or ambient.
13. Do not write generic rationales; explain the target-specific bond formation and functional group compatibility.
14. If optimizing for yield, do not claim a route has the highest yield unless its expected_yield_percent is equal to or higher than the other returned routes.
15. The overall_recommendation must name one of the returned routes and must not contradict any route objective_fit text.

JSON STRUCTURE:

{
  "routes": [
    {
      "route_name": "Short descriptive route name",
      "summary": "Why this route is distinct from the other options",
      "steps": [
        {
          "reaction_name": "Named or descriptive reaction",
          "reactants": ["SMILES_A", "SMILES_B"],
          "product_smiles": "SMILES_PRODUCT",
          "stoichiometry": "Example: A 1.0 equiv, B 1.2 equiv, base 2.0 equiv",
          "reagents": "Catalysts, bases, acids, oxidants, additives",
          "solvent": "Solvent or solvent mixture",
          "temperature_celsius": "temperature or range with units, e.g. 80 deg C or reflux",
          "reaction_time": "time or range with units, e.g. 6 h or overnight",
          "atmosphere": "air, nitrogen, argon, oxygen-free, etc.",
          "workup_purification": "Quench, extraction, chromatography, crystallization, etc.",
          "expected_yield_percent": "estimated percent or range with percent sign",
          "important_conditions": "Other important operational details",
          "rationale": "Why this single reaction forms the target product",
          "objective_fit": "How this route addresses the selected optimization objective without contradicting the expected yield or other routes",
          "evidence_reaction_ids": ["reaction_id_1", "reaction_id_2"]
        }
      ],
      "objective_fit": "How this route addresses the selected optimization objective without contradicting the expected yield or other routes",
      "evidence_reaction_ids": ["reaction_id_1", "reaction_id_2"]
    }
  ],
  "overall_recommendation": "Which returned route is preferred and why, consistent with objective_fit and expected yields"
}
""".strip()


def build_retro_planner_messages(target_smiles: str):
    user_prompt = f"""Target Molecule SMILES: {target_smiles}
Every route must contain exactly one reaction step.
Every route's only step product_smiles must be this exact target molecule.
Generate exactly 1 distinct route option."""

    return [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]


messages = build_retro_planner_messages(TARGET_SMILES)

output = llm.create_chat_completion(
    messages=messages,
    temperature=0.1,
    max_tokens=2048,
    response_format={
        "type": "json_object"
    }
)

raw_text = output["choices"][0]["message"]["content"].strip()

print("RAW MODEL OUTPUT:")
print(raw_text)

Llama.generate: 780 prefix-match hit, remaining 52 prompt tokens to eval
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled

RAW MODEL OUTPUT:
{
  "routes": [
    {
      "route_name": "Direct Nitration",
      "summary": "A direct nitration of aniline to form nitrobenzene",
      "steps": [
        {
          "reaction_name": "Nitration of aniline",
          "reactants": ["c1ccccc1N", "[O-]N=[O+]O"],
          "product_smiles": "O=[N+]([O-])c1ccccc1",
          "stoichiometry": "Aniline 1.0 equiv, [O-]N=[O+]O 1.0 equiv",
          "reagents": "Conc. H2SO4, HNO3",
          "solvent": "Conc. H2SO4",
          "temperature_celsius": "0 deg C",
          "reaction_time": "1 h",
          "atmosphere": "air",
          "workup_purification": "Quench with ice, extract with EtOAc, wash with water, dry over Na2SO4, concentrate",
          "expected_yield_percent": "80",
          "important_conditions": "Anhydrous conditions",
          "rationale": "Aniline is nitrated to form nitrobenzene in a single step.",
          "objective_fit": "This route is the most direct and efficient way to form nitrobenzene from a

In [31]:
import json

TARGET_SMILES = "CC(C)(O)c1ccccc1"

SYSTEM_PROMPT = """
You are an expert Organic Chemist specializing in Retrosynthesis.
Your task is to propose practical alternative forward synthesis routes that make the given Target Molecule.

CRITICAL RULES:
1. Output MUST be valid JSON only.
2. Use standard SMILES strings for all molecules.
3. Verify that each route can plausibly form the stated final product.
4. Do not output Markdown formatting like ```json.
5. Use literature-like reaction conditions. If exact values are uncertain, provide realistic approximate ranges.
6. Generate exactly 1 distinct route option.
7. Each route must contain exactly one reaction step in its steps list. Do not propose intermediates, precursor syntheses, or multi-step sequences.
8. The only step product_smiles in every route MUST be the target molecule SMILES after canonicalization.
9. Each route must be genuinely distinct: different disconnection, starting material class, or key reaction type.
10. Avoid ambiguous reagent descriptions: if SMILES CO is methanol, call it methanol or MeOH in conditions; only use carbon monoxide when the intended reagent is truly carbon monoxide.
11. The named or descriptive reaction must include the catalysts, activators, bases, acids, or other reagents that make the stated chemistry plausible.
12. Temperature values must include units or clear descriptors such as room temperature, reflux, or ambient.
13. Do not write generic rationales; explain the target-specific bond formation and functional group compatibility.
14. If optimizing for yield, do not claim a route has the highest yield unless its expected_yield_percent is equal to or higher than the other returned routes.
15. The overall_recommendation must name one of the returned routes and must not contradict any route objective_fit text.

JSON STRUCTURE:

{
  "routes": [
    {
      "route_name": "Short descriptive route name",
      "summary": "Why this route is distinct from the other options",
      "steps": [
        {
          "reaction_name": "Named or descriptive reaction",
          "reactants": ["SMILES_A", "SMILES_B"],
          "product_smiles": "SMILES_PRODUCT",
          "stoichiometry": "Example: A 1.0 equiv, B 1.2 equiv, base 2.0 equiv",
          "reagents": "Catalysts, bases, acids, oxidants, additives",
          "solvent": "Solvent or solvent mixture",
          "temperature_celsius": "temperature or range with units, e.g. 80 deg C or reflux",
          "reaction_time": "time or range with units, e.g. 6 h or overnight",
          "atmosphere": "air, nitrogen, argon, oxygen-free, etc.",
          "workup_purification": "Quench, extraction, chromatography, crystallization, etc.",
          "expected_yield_percent": "estimated percent or range with percent sign",
          "important_conditions": "Other important operational details",
          "rationale": "Why this single reaction forms the target product",
          "objective_fit": "How this route addresses the selected optimization objective without contradicting the expected yield or other routes",
          "evidence_reaction_ids": ["reaction_id_1", "reaction_id_2"]
        }
      ],
      "objective_fit": "How this route addresses the selected optimization objective without contradicting the expected yield or other routes",
      "evidence_reaction_ids": ["reaction_id_1", "reaction_id_2"]
    }
  ],
  "overall_recommendation": "Which returned route is preferred and why, consistent with objective_fit and expected yields"
}
""".strip()


def build_retro_planner_messages(target_smiles: str):
    user_prompt = f"""Target Molecule SMILES: {target_smiles}
Every route must contain exactly one reaction step.
Every route's only step product_smiles must be this exact target molecule.
Generate exactly 1 distinct route option."""

    return [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]


messages = build_retro_planner_messages(TARGET_SMILES)

output = llm.create_chat_completion(
    messages=messages,
    temperature=0.1,
    max_tokens=2048,
    response_format={
        "type": "json_object"
    }
)

raw_text = output["choices"][0]["message"]["content"].strip()

print("RAW MODEL OUTPUT:")
print(raw_text)

Llama.generate: 780 prefix-match hit, remaining 48 prompt tokens to eval
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled

RAW MODEL OUTPUT:
{
  "routes": [
    {
      "route_name": "Alcohol Protection",
      "summary": "Protect the alcohol group of 2-methylphenol with a protecting group to enable further functionalization.",
      "steps": [
        {
          "reaction_name": "Methylation of 2-methylphenol",
          "reactants": ["COC1=CC=CC=C1C"],
          "product_smiles": "CC(C)(O)c1ccccc1",
          "stoichiometry": "COC1=CC=CC=C1C 1.0 equiv",
          "reagents": "Methyliodide",
          "solvent": "Acetone",
          "temperature_celsius": "0 deg C",
          "reaction_time": "1 h",
          "atmosphere": "N2",
          "workup_purification": "Quench with aqueous Na2S2O3, extract with EtOAc, dry over Na2SO4, concentrate in vacuo",
          "expected_yield_percent": "80",
          "important_conditions": "Methyliodide is generated in situ from CH3I and NaH.",
          "rationale": "Methyliodide is a good electrophile that can be used to protect the alcohol group of 2-methylphenol. Th

In [32]:
import json

TARGET_SMILES = "c1ccc(-c2ccccc2)cc1"

SYSTEM_PROMPT = """
You are an expert Organic Chemist specializing in Retrosynthesis.
Your task is to propose practical alternative forward synthesis routes that make the given Target Molecule.

CRITICAL RULES:
1. Output MUST be valid JSON only.
2. Use standard SMILES strings for all molecules.
3. Verify that each route can plausibly form the stated final product.
4. Do not output Markdown formatting like ```json.
5. Use literature-like reaction conditions. If exact values are uncertain, provide realistic approximate ranges.
6. Generate exactly 1 distinct route option.
7. Each route must contain exactly one reaction step in its steps list. Do not propose intermediates, precursor syntheses, or multi-step sequences.
8. The only step product_smiles in every route MUST be the target molecule SMILES after canonicalization.
9. Each route must be genuinely distinct: different disconnection, starting material class, or key reaction type.
10. Avoid ambiguous reagent descriptions: if SMILES CO is methanol, call it methanol or MeOH in conditions; only use carbon monoxide when the intended reagent is truly carbon monoxide.
11. The named or descriptive reaction must include the catalysts, activators, bases, acids, or other reagents that make the stated chemistry plausible.
12. Temperature values must include units or clear descriptors such as room temperature, reflux, or ambient.
13. Do not write generic rationales; explain the target-specific bond formation and functional group compatibility.
14. If optimizing for yield, do not claim a route has the highest yield unless its expected_yield_percent is equal to or higher than the other returned routes.
15. The overall_recommendation must name one of the returned routes and must not contradict any route objective_fit text.

JSON STRUCTURE:

{
  "routes": [
    {
      "route_name": "Short descriptive route name",
      "summary": "Why this route is distinct from the other options",
      "steps": [
        {
          "reaction_name": "Named or descriptive reaction",
          "reactants": ["SMILES_A", "SMILES_B"],
          "product_smiles": "SMILES_PRODUCT",
          "stoichiometry": "Example: A 1.0 equiv, B 1.2 equiv, base 2.0 equiv",
          "reagents": "Catalysts, bases, acids, oxidants, additives",
          "solvent": "Solvent or solvent mixture",
          "temperature_celsius": "temperature or range with units, e.g. 80 deg C or reflux",
          "reaction_time": "time or range with units, e.g. 6 h or overnight",
          "atmosphere": "air, nitrogen, argon, oxygen-free, etc.",
          "workup_purification": "Quench, extraction, chromatography, crystallization, etc.",
          "expected_yield_percent": "estimated percent or range with percent sign",
          "important_conditions": "Other important operational details",
          "rationale": "Why this single reaction forms the target product",
          "objective_fit": "How this route addresses the selected optimization objective without contradicting the expected yield or other routes",
          "evidence_reaction_ids": ["reaction_id_1", "reaction_id_2"]
        }
      ],
      "objective_fit": "How this route addresses the selected optimization objective without contradicting the expected yield or other routes",
      "evidence_reaction_ids": ["reaction_id_1", "reaction_id_2"]
    }
  ],
  "overall_recommendation": "Which returned route is preferred and why, consistent with objective_fit and expected yields"
}
""".strip()


def build_retro_planner_messages(target_smiles: str):
    user_prompt = f"""Target Molecule SMILES: {target_smiles}
Every route must contain exactly one reaction step.
Every route's only step product_smiles must be this exact target molecule.
Generate exactly 1 distinct route option."""

    return [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]


messages = build_retro_planner_messages(TARGET_SMILES)

output = llm.create_chat_completion(
    messages=messages,
    temperature=0.1,
    max_tokens=2048,
    response_format={
        "type": "json_object"
    }
)

raw_text = output["choices"][0]["message"]["content"].strip()

print("RAW MODEL OUTPUT:")
print(raw_text)

Llama.generate: 780 prefix-match hit, remaining 51 prompt tokens to eval
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled

RAW MODEL OUTPUT:
{
  "routes": [
    {
      "route_name": "Direct Arylation",
      "summary": "Direct arylation of benzene with iodobenzene using a palladium catalyst",
      "steps": [
        {
          "reaction_name": "Direct arylation of benzene with iodobenzene using a palladium catalyst",
          "reactants": ["C1=CC=C(C=C1)I", "C1=CC=CC=C1"],
          "product_smiles": "c1ccc(-c2ccccc2)cc1",
          "stoichiometry": "C1=CC=C(C=C1)I 1.0 equiv, C1=CC=CC=C1 1.0 equiv, base 2.0 equiv",
          "reagents": "Pd(PPh3)4, K2CO3, CuI",
          "solvent": "DMF",
          "temperature_celsius": "100 deg C",
          "reaction_time": "12 h",
          "atmosphere": "N2",
          "workup_purification": "Quench, extraction, chromatography, crystallization, etc.",
          "expected_yield_percent": "70",
          "important_conditions": "The reaction is performed under an inert atmosphere.",
          "rationale": "The reaction is performed under an inert atmosphere.",
     

In [33]:
import sys
import os
import json

# Вимикаємо вивід низькорівневих логів C++ бібліотеки в stderr
sys.stderr = open(os.devnull, 'w')
from llama_cpp import Llama
sys.stderr = sys.__stderr__ # Повертаємо стандартний потік назад

SYSTEM_PROMPT = """
You are an expert Organic Chemist specializing in Retrosynthesis.
Your task is to propose practical alternative forward synthesis routes that make the given Target Molecule.

CRITICAL RULES:
1. Output MUST be valid JSON only.
2. Use standard SMILES strings for all molecules.
3. Verify that each route can plausibly form the stated final product.
4. Do not output Markdown formatting like ```json.
5. Use literature-like reaction conditions. If exact values are uncertain, provide realistic approximate ranges.
6. Generate exactly 1 distinct route option.
7. Each route must contain exactly one reaction step in its steps list. Do not propose intermediates, precursor syntheses, or multi-step sequences.
8. The only step product_smiles in every route MUST be the target molecule SMILES after canonicalization.
9. Each route must be genuinely distinct: different disconnection, starting material class, or key reaction type.
10. Avoid ambiguous reagent descriptions: if SMILES CO is methanol, call it methanol or MeOH in conditions; only use carbon monoxide when the intended reagent is truly carbon monoxide.
11. The named or descriptive reaction must include the catalysts, activators, bases, acids, or other reagents that make the stated chemistry plausible.
12. Temperature values must include units or clear descriptors such as room temperature, reflux, or ambient.
13. Do not write generic rationales; explain the target-specific bond formation and functional group compatibility.
14. If optimizing for yield, do not claim a route has the highest yield unless its expected_yield_percent is equal to or higher than the other returned routes.
15. The overall_recommendation must name one of the returned routes and must not contradict any route objective_fit text.

JSON STRUCTURE:
{
  "routes": [
    {
      "route_name": "Short descriptive route name",
      "summary": "Why this route is distinct from the other options",
      "steps": [
        {
          "reaction_name": "Named or descriptive reaction",
          "reactants": ["SMILES_A", "SMILES_B"],
          "product_smiles": "SMILES_PRODUCT",
          "stoichiometry": "Example: A 1.0 equiv, B 1.2 equiv, base 2.0 equiv",
          "reagents": "Catalysts, bases, acids, oxidants, additives",
          "solvent": "Solvent or solvent mixture",
          "temperature_celsius": "temperature or range with units, e.g. 80 deg C or reflux",
          "reaction_time": "time or range with units, e.g. 6 h or overnight",
          "atmosphere": "air, nitrogen, argon, oxygen-free, etc.",
          "workup_purification": "Quench, extraction, chromatography, crystallization, etc.",
          "expected_yield_percent": "estimated percent or range with percent sign",
          "important_conditions": "Other important operational details",
          "rationale": "Why this single reaction forms the target product",
          "objective_fit": "How this route addresses the selected optimization objective without contradicting the expected yield or other routes",
          "evidence_reaction_ids": ["reaction_id_1", "reaction_id_2"]
        }
      ],
      "objective_fit": "How this route addresses the selected optimization objective without contradicting the expected yield or other routes",
      "evidence_reaction_ids": ["reaction_id_1", "reaction_id_2"]
    }
  ],
  "overall_recommendation": "Which returned route is preferred and why, consistent with objective_fit and expected yields"
}
""".strip()

# Ініціалізація моделі (виконується один раз, verbose=False вимикає частину логів)
llm = Llama.from_pretrained(
    repo_id="mradermacher/ChemLLM-20B-Chat-SFT-i1-GGUF",
    filename="ChemLLM-20B-Chat-SFT.i1-Q4_K_M.gguf",
    n_ctx=2048,
    n_gpu_layers=25,
    verbose=False
)

def get_retrosynthesis(target_smiles: str) -> str:
    """Приймає TARGET_SMILES, викликає модель і повертає чистий JSON рядок."""
    user_prompt = f"""Target Molecule SMILES: {target_smiles}
Every route must contain exactly one reaction step.
Every route's only step product_smiles must be this exact target molecule.
Generate exactly 1 distinct route option."""

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ]

    output = llm.create_chat_completion(
        messages=messages,
        temperature=0.1,
        max_tokens=1500,
        response_format={"type": "json_object"}
    )

    return output["choices"][0]["message"]["content"].strip()

In [34]:
TARGET_SMILES = "O=C(/C=C/c1ccccc1)c1ccccc1" # 5
raw_json = get_retrosynthesis(TARGET_SMILES)

print("RAW MODEL OUTPUT:")
print(raw_json)

RAW MODEL OUTPUT:
{
  "routes": [
    {
      "route_name": "Aldol Condensation",
      "summary": "Aldol condensation of benzaldehyde with acetophenone",
      "steps": [
        {
          "reaction_name": "Aldol condensation",
          "reactants": ["C1=CC=CC=C1C=O", "C1=CC=CC=C1C(=O)C1=CC=CC=C1"],
          "product_smiles": "O=C(/C=C/c1ccccc1)c1ccccc1",
          "stoichiometry": "A 1.0 equiv, B 1.0 equiv, base 1.0 equiv",
          "reagents": "NaOAc, H2O",
          "solvent": "CH3COOH",
          "temperature_celsius": "25.0 degrees Celsius",
          "reaction_time": "1.0 hours",
          "atmosphere": "air",
          "workup_purification": "Crystallization",
          "expected_yield_percent": "70.0",
          "important_conditions": "The reaction is performed in aqueous solution to facilitate the formation of the aldol product.",
          "rationale": "The aldol condensation of benzaldehyde with acetophenone is a classic example of a carbon-carbon bond formation react

In [ ]:
TARGET_SMILES = "C=Cc1ccccc1" # 6
raw_json = get_retrosynthesis(TARGET_SMILES)

print("RAW MODEL OUTPUT:")
print(raw_json)

RAW MODEL OUTPUT:
{
  "routes": [
    {
      "route_name": "Stille Coupling",
      "summary": "A classic cross-coupling reaction for the formation of C=Cc1ccccc1",
      "steps": [
        {
          "reaction_name": "Stille Coupling",
          "reactants": ["Brc1ccccc1", "C=CB(O)O"],
          "product_smiles": "C=Cc1ccccc1",
          "stoichiometry": "Brc1ccccc1 1.0 equiv, C=CB(O)O 1.0 equiv",
          "reagents": "Pd(PPh3)4, CuI, PPh3",
          "solvent": "DMF",
          "temperature_celsius": 100,
          "reaction_time": 12,
          "atmosphere": "N2",
          "workup_purification": "Purification by column chromatography",
          "expected_yield_percent": 80,
          "important_conditions": "The reaction is performed under an inert atmosphere.",
          "rationale": "The Stille coupling reaction is a classic cross-coupling reaction that allows the formation of C=Cc1ccccc1 from Brc1ccccc1 and C=CB(O)O. The reaction is catalyzed by Pd(PPh3)4 and CuI, and PPh3 i

In [ ]:
TARGET_SMILES = "C1=CCCCC1" # 7
raw_json = get_retrosynthesis(TARGET_SMILES)

print("RAW MODEL OUTPUT:")
print(raw_json)

RAW MODEL OUTPUT:
{
  "routes": [
    {
      "route_name": "Cyclohexane from Cyclohexene",
      "summary": "Cyclohexene is hydrogenated to cyclohexane",
      "steps": [
        {
          "reaction_name": "Hydrogenation",
          "reactants": ["C1=CCCCC1"],
          "product_smiles": "C1CCCCC1",
          "stoichiometry": "1.0 equiv",
          "reagents": "Palladium on carbon",
          "solvent": "Ethanol",
          "temperature_celsius": "25 deg C",
          "reaction_time": "12 h",
          "atmosphere": "Hydrogen",
          "workup_purification": "Filter and concentrate",
          "expected_yield_percent": "90",
          "important_conditions": "The reaction is carried out under an atmosphere of hydrogen.",
          "rationale": "Cyclohexene is hydrogenated to cyclohexane using palladium on carbon as the catalyst.",
          "objective_fit": "This route is the most efficient and practical for the synthesis of cyclohexane from cyclohexene.",
          "evidence_reac

In [ ]:
TARGET_SMILES = "CC(O)c1ccccc1" # 8
raw_json = get_retrosynthesis(TARGET_SMILES)

print("RAW MODEL OUTPUT:")
print(raw_json)

RAW MODEL OUTPUT:
{
  "routes": [
    {
      "route_name": "Alcohol Formation",
      "summary": "Direct formation of the target molecule from the starting material",
      "steps": [
        {
          "reaction_name": "Reduction of aldehyde",
          "reactants": ["CC=O"],
          "product_smiles": "CC(O)c1ccccc1",
          "stoichiometry": "CC=O 1.0 equiv",
          "reagents": "NaBH4",
          "solvent": "THF",
          "temperature_celsius": "0 deg C",
          "reaction_time": "1 h",
          "atmosphere": "N2",
          "workup_purification": "Quench with water, extract with EtOAc, dry over Na2SO4, concentrate",
          "expected_yield_percent": "90",
          "important_conditions": "Anhydrous conditions",
          "rationale": "The target molecule is formed by the reduction of the aldehyde intermediate with sodium borohydride in tetrahydrofuran at 0°C.",
          "objective_fit": "This route directly forms the target molecule with a good yield, making it sui

In [ ]:
TARGET_SMILES = "O=C1OC(=O)C2CC=CCC12" # 9
raw_json = get_retrosynthesis(TARGET_SMILES)

print("RAW MODEL OUTPUT:")
print(raw_json)

RAW MODEL OUTPUT:
{
  "routes": [
    {
      "route_name": "Direct Oxidation",
      "summary": "A direct oxidation of 1,2,3,4-tetrahydronaphthalene to the corresponding lactone",
      "steps": [
        {
          "reaction_name": "Oxidation",
          "reactants": ["C1CCC2C1C(=O)O2"],
          "product_smiles": "O=C1OC(=O)C2CC=CCC12",
          "stoichiometry": "1.0 equiv",
          "reagents": "O3",
          "solvent": "CC(=O)O",
          "temperature_celsius": 25.0,
          "reaction_time": 2.0,
          "atmosphere": "Oxygen",
          "workup_purification": "Crystallization",
          "expected_yield_percent": 80.0,
          "important_conditions": "The reaction is carried out in a sealed tube under an oxygen atmosphere.",
          "rationale": "The direct oxidation of 1,2,3,4-tetrahydronaphthalene to the corresponding lactone is a well-established reaction. The reaction is carried out in a sealed tube under an oxygen atmosphere to ensure the formation of the desir

In [ ]:
TARGET_SMILES = "CCNC(C)=O" # 10
raw_json = get_retrosynthesis(TARGET_SMILES)

print("RAW MODEL OUTPUT:")
print(raw_json)